In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit, expit
from typing import List, Dict, Any, Tuple, Optional

from bayesian_inference import BayesianCalibration

In [ ]:
ANNOT_PATH = "../exp_readmission/_output/gpt-4o-2024-08-06/prompt_v1/limit_1/annotations.csv"

In [ ]:
bc = BayesianCalibration(stan_model_path="bayesian_logistic_regression.stan")
bc.compile_model()

In [ ]:
true_beta_0, true_beta_1 = 0, 1
def generate_test_data(n_samples: int = 50, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """
    Generate synthetic test data for calibration analysis.

    Args:
        n_samples: Number of samples to generate
        seed: Random seed for reproducibility

    Returns:
        Tuple of (confidences, annotations)
    """
    np.random.seed(seed)

    # Generate confidences uniformly
    confidences = np.random.uniform(0.5, 0.95, n_samples)

    # Generate annotations with some calibration relationship
    logit_confidences = logit(confidences)
    true_probs = expit(true_beta_0 + true_beta_1 * logit_confidences)

    annotations = np.random.binomial(1, true_probs)

    return confidences, annotations

In [ ]:
# confidences, annotations = generate_test_data(100)

In [ ]:
annot_df = pd.read_csv(ANNOT_PATH)
annot_df = annot_df[annot_df.feedback_type == "reason"]
confidences = annot_df.LLM_confidence.to_numpy().astype(float)
annotations = annot_df.annotation.to_numpy().astype(int)
pd.DataFrame({
    "conf": confidences,
    "annot": annotations
})

In [ ]:
results = bc.fit(
            confidences,
            annotations,
            chains=2,
            iter_sampling=500,
            iter_warmup=500,
            show_progress=True
        )

In [ ]:
bc.plot_calibration_curves(
    max_curves=50,
    show_individual_curves=True,
    # true_beta_0=true_beta_0,
    # true_beta_1=true_beta_1
)